# Gradio User Interface

In [ ]:
import gradio as gr

# Closing all open ports
gr.close_all()

In [ ]:
import gradio as gr
import pandas as pd

# Beispiel-DataFrame
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Alter": [25, 30, 35],
    "Beruf": ["Ingenieur", "Lehrer", "Designer"]
})

# Interface-Funktion, gibt das DataFrame zurück
def show_df():
    return df

app = gr.Interface(fn=show_df, inputs=[], outputs=gr.Dataframe())

app.launch()


In [ ]:
import gradio as gr


def greet(name):
    return "Hello " + name + "!"


with gr.Blocks() as demo:
    title = gr.HTML("<div style=font-size:100px;'>Stock Predictor</div>")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            load = gr.Button("Load Stock")
            load = gr.Button("Loading Bar")
            #load.click(fn=greet, inputs=name, outputs=output)
            result = gr.Label(value="Successful", label="Result")
        with gr.Column(scale=4):
            img1 = gr.Image("google-test.png")

    with gr.Row():
        with gr.Column(scale=1):
            latestdata = gr.Label(value="Latest Datas of Stock: 2025-06-11", label="Currency")
            nobrands = gr.Label(value="Number of Brands: 61", label="Countin Brands")
            #output = gr.Textbox(label="Output Box")
            #name = gr.Textbox(label="Name")
            drpdwn = gr.Dropdown(label="Choose your Brand:", choices=["Google", "BMW", "Apple"], allow_custom_value=True)
            slider = gr.Slider(label="Number of Days to predict", maximum=10, step=1, value=4)
            prdct = gr.Button("Predict")
        with gr.Column(scale=1):
            start = gr.Label(value="Start Stock in Dataset: 38,343984", label="Start")
            low = gr.Label(value="Lowest Stock: 21,98733", label="Low")
            high = gr.Label(value="Highest Stock: 358,546202", label="High")
            latest = gr.Label(value="Latest Stock: 298,043746", label="Latest")
    news = gr.HTML("<div style=font-size:40px;'>The Latest news of Google:</div>")
    news1 = gr.Label(value="Googles KI wird Modeberater, aber kein Ersatz für Apps", label="Spiegel")
    news2 = gr.Label(value="Google hat jetzt ein neues App-Icon", label="FAZ")
    news3 = gr.Label(value="Mexiko verklagt Google wegen >>Golf von Amerika<<", label="Bild.de")

    #greet_btn = gr.Button("Predict")
    #greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()


In [22]:
gr.close_all()

# Prototype mit Funktionen

In [6]:
import pandas as pd
import kagglehub
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
#import os.path
from pathlib import Path
import time
import gradio as gr

import numpy as np
from io import BytesIO
from PIL import Image

In [ ]:
#Als Datenrespresentation für die Stocks nehmen?

def image_classifier(inp):
    return {'cat': 0.3, 'dog': 0.7, "viech": 0.9}

demo = gr.Interface(fn=image_classifier, inputs="image", outputs="label")
demo.launch()

In [ ]:
def download(): #unten in den Code noch implementieren
    global path
    global brands
    try: #try download, if fail give the except-statement back
        path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating") #Download latest version
        print("Path to dataset files:", path)
        erg = "<div style=font-size:20px;><div style=background-color:green;><center>Download successul</center></div></div>"

        #Load Data in pandas
        data_path = path+"\World-Stock-Prices-Dataset.csv"
        stockdata = pd.read_csv(data_path)
        #Create DataFrame
        df = pd.DataFrame(stockdata)

        brands = df["Brand_Name"].unique().tolist() #create a list of all brands in the df
        brand_AUSWAHL = brands[39] #Platzhalter für eine spätere Auswahl vom User
        df_apple = df.loc[df["Brand_Name"] == brand_AUSWAHL, ["Date", "Close", "Brand_Name"]]
    except:
        erg = "<div style=font-size:20px;><div style=background-color:red;><center>Download failed</center></div></div>"
        #df = pd.DataFrame({"Fail": ["Fail"]})
    return erg

In [22]:
def updatedata():
    path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")
    return "<div style=font-size:20px;><div style=background-color:green;><center>Update Successul</center></div></div>"



def predict(brand, days):
    df_brand = df.loc[df["Brand_Name"] == brand, ["Date", "Close", "Brand_Name"]]
    brandname = "<div style=font-size:40px;'>"+brand+"</div>"

    brand_first = str(df_brand["Close"].iloc[-1])
    brand_latest = str(df_brand["Close"].iloc[0])
    max_stock = df_brand["Close"].max()
    min_stock = df_brand["Close"].min()


    df_brand_preproc = df_brand.rename(columns={"Date": "timestamp", "Close": "target", "Brand_Name": "item_id"})
    df_brand_preproc["item_id"] = df_brand_preproc['item_id'].astype("string")
    df_brand_preproc["timestamp"] = df_brand_preproc['timestamp'].astype("string")
    timecut = df_brand_preproc["timestamp"].str.slice(stop=10) #Cut hh:mm:ss and timezone
    df_brand_preproc["timestamp"] = timecut
    df_brand_preproc["timestamp"] = pd.to_datetime(timecut) #convert string into datetime64
    df_brand_reordered =  df_brand_preproc[['item_id', 'timestamp', 'target']] #Reordering columns
    df_irregular = TimeSeriesDataFrame(
        pd.DataFrame(df_brand_reordered)
    )
    df_regular = df_irregular.convert_frequency(freq="D")
    df_filled = df_regular.fill_missing_values()
    data = TimeSeriesDataFrame.from_data_frame(
        df = df_filled,
        id_column="item_id",
        timestamp_column="timestamp"
    )

    prediction_length = days
    train_data, test_data = data.train_test_split(prediction_length)

    predictor = TimeSeriesPredictor(prediction_length=prediction_length, freq="D").fit(
        train_data, presets="bolt_base", hyperparameters={"Chronos": {"fine_tune": True, "fine_tune_lr": 1e-3, "fine_tune_steps": 2000}},
        time_limit=3,
    )

    predictions = predictor.predict(train_data)
    prdct_img = predictor.plot(
        data=data,
        predictions=predictions,
        item_ids=data.item_ids[:2],
        max_history_length=200,
    );

    buf = BytesIO()
    prdct_img.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    image = Image.open(buf) 

    return brand_first, brand_latest, brandname, max_stock, min_stock, np.array(image)

with gr.Blocks() as demo:
    #Download latest version
    path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")

    #Load Data in pandas
    data_path = path+"\World-Stock-Prices-Dataset.csv"
    stockdata = pd.read_csv(data_path)
    #Create DataFrame
    df = pd.DataFrame(stockdata)

    date_clean = df["Date"].str.slice(stop=10)
    date_first = date_clean.iloc[-1]
    date_latest = date_clean.iloc[0]

    brands = df["Brand_Name"].unique().tolist() #create a list of all brands in the df

    
    title = gr.HTML("<div style=font-size:100px;'>Stock Predictor</div>")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            load = gr.Button("Update Stock")
            dataupdate = gr.HTML("<div style=font-size:20px;><div style=background-color:grey;><center>Data not updated yet</center></div></div>")
            load.click(fn=updatedata, inputs=[], outputs=dataupdate)
        with gr.Column(scale=4):
            df_presentator = df[["Date","Close","Brand_Name","Country"]]
            print("HIER")
            print(df_presentator)
            print("HIER ENDE")
            gr.DataFrame(df_presentator)

    with gr.Row():
        with gr.Column(scale=1):
            nobrands = gr.Label(value="Number of Brands in Dataset: "+str(len(brands)), label="Counting Brands")
            #output = gr.Textbox(label="Output Box")
            #name = gr.Textbox(label="Name")
            
        with gr.Column(scale=1):
            latestdata = gr.Label(value="Latest Data of Stock: "+date_latest, label="Currency")

    drpdwn = gr.Dropdown(label="Choose your Brand:", choices=brands, interactive=True)
    slider = gr.Slider(label="Number of Days to predict", minimum=1, maximum=50, step=1, value=3, interactive=True)
    prdct = gr.Button("Predict")
    
    brand_titel = gr.HTML()
    img1 = gr.Image("google-test.png")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            start = gr.Label(value="Start Stock in Dataset", label="Start Stock")
            low = gr.Label(value="Lowest Stock", label="Historically Lowest Stock")
            
        with gr.Column(scale=1):
            latest = gr.Label(value="Latest Stock", label="Current Stock")
            high = gr.Label(value="Highest Stock", label="Historically Highest Stock")
    
    prdct.click(fn=predict, inputs=[drpdwn, slider], outputs=[start, latest, brand_titel, high, low, img1])
    news = gr.HTML("<div style=font-size:40px;'>The Latest news of Google:</div>")
    news1 = gr.Label(value="Googles KI wird Modeberater, aber kein Ersatz für Apps", label="Spiegel")
    news2 = gr.Label(value="Google hat jetzt ein neues App-Icon", label="FAZ")
    news3 = gr.Label(value="Mexiko verklagt Google wegen >>Golf von Amerika<<", label="Bild.de")

    #greet_btn = gr.Button("Predict")
    #greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()

<>:63: SyntaxWarning: invalid escape sequence '\W'
<>:63: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_7688\3074649756.py:63: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"


HIER
                             Date       Close                 Brand_Name  \
0       2025-06-24 00:00:00-04:00    6.590000                    peloton   
1       2025-06-24 00:00:00-04:00   99.120003                      crocs   
2       2025-06-24 00:00:00-04:00   70.209999      the coca-cola company   
3       2025-06-24 00:00:00-04:00  115.809998                     adidas   
4       2025-06-24 00:00:00-04:00  308.380005           american express   
...                           ...         ...                        ...   
309596  2000-01-03 00:00:00-05:00   19.158504                    philips   
309597  2000-01-03 00:00:00-05:00   14.781797      the coca-cola company   
309598  2000-01-03 00:00:00-05:00    9.467610         southwest airlines   
309599  2000-01-03 00:00:00-05:00   23.075401                     target   
309600  2000-01-03 00:00:00-05:00    5.482471  american eagle outfitters   

            Country  
0               usa  
1               usa  
2               